# Pré-Modelagem — Balneabilidade (versão corrigida)

**Autor:** Wagner Oliveira · **Turma:** T9 · **Orientador:** Prof. Jorge Luiz Bezerra de Araújo

Esse notebook é uma atualização do `laudos_antigos copy.ipynb`, que é a base do meu TCC.
Enquanto eu revisava o texto final, achei um bug no jeito que eu juntava os dados de clima do
INMET com os laudos da SEMACE — e esse bug tava inflando a base e causando vazamento de dados
entre treino e teste. Explico com calma na Seção 3.

Mantive toda a lógica original (limpeza, fuzzy matching, seleção de variáveis, EDA e o
treinamento em 3 fases) exatamente como estava. A única coisa que mudou de verdade é a
correção, que isolei numa seção própria pra ficar fácil de ver o antes e o depois.


### Resolução CONAMA 274

Essas colunas não são apenas números ou textos ruidosos; elas representam a "impressão digital" da qualidade da água. Na legislação brasileira (como a Resolução CONAMA 274 que trata de balneabilidade), cada um desses parâmetros funciona como um indicador de um tipo específico de poluição ou risco à saúde humana.

Aqui está o porquê de cada uma ser um pilar fundamental para o seu modelo:

1. Coliformes Termotolerantes (O Vilão Biológico)
Este é o parâmetro mais crítico para a balneabilidade.

    * O que indica: Presença de fezes de animais de sangue quente (humanos ou animais).
    * Risco: Indica a probabilidade de existirem microrganismos patogénicos (vírus, bactérias e protozoários) que causam doenças como gastroenterite, hepatite A e cólera.
    * Para o modelo: É quase sempre o fator determinante. Se os valores de "Presença" ou os números extraídos pela nossa função forem altos, a água é automaticamente Imprópria

2. Turbidez e Cor (Os Indicadores Físicos)
Embora pareçam apenas questões estéticas, elas escondem perigos invisíveis.

    * Turbidez: Mede a dificuldade de a luz atravessar a água devido a partículas em suspensão (argila, algas, detritos). Muita turbidez pode "proteger" as bactérias da radiação UV (que as mataria naturalmente) e dificultar a desinfecção.
    * Cor: Geralmente indica a presença de matéria orgânica dissolvida ou metais.
    * Para o modelo: Funcionam como "sinais de alerta". Se a cor está "Presente" ou a turbidez é alta, há uma forte correlação com a presença de poluentes químicos ou biológicos.

3. Nitrato e Nitrogênio Amoniacal Total (Os Indicadores Químicos)
Estes parâmetros revelam o histórico de poluição daquela água.

    * Nitrogênio Amoniacal: Indica poluição recente. A amónia é um dos primeiros produtos da decomposição de esgotos domésticos ou resíduos industriais.
    * Nitrato: Indica poluição antiga ou persistente, pois é o estágio final da oxidação do nitrogênio. Também pode vir do escoamento de fertilizantes agrícolas.
    * Para o modelo: Ajudam a distinguir se a contaminação é um evento agudo (um vazamento de esgoto agora) ou um problema crónico daquela região.

4. Nitrito

    * O que indica: É um estágio intermediário entre a amónia e o nitrato. É muito instável e altamente tóxico para a vida aquática.
    * Para o modelo: A presença de Nitrito (especialmente se marcado como "Presença" nas suas tags) é um sinal de que o processo de autodepuração da água (é um processo natural de recuperação de rios e lagos, onde microrganismos decompõem a matéria orgânica (esgoto) e reequilibram os níveis de oxigênio após a poluição) ainda não terminou, indicando perigo imediato.


## 1. Bibliotecas

In [ ]:
import pandas as pd
import re
import csv
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Padrão visual: fontes maiores e consistentes em todas as figuras do notebook,
# atendendo à observação do professor de que os gráficos no padrão default do
# matplotlib/seaborn ficam pequenos demais em relação ao texto do TCC.
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 13,
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 12,
    'font.family': 'DejaVu Sans',
})


## 2. Importação do dataset e limpeza inicial

Essa parte não mudou nada em relação à análise original: carrego o Excel dos laudos, tiro as
colunas que não servem pra análise (IDs, referência pessoal, colunas de controle), arrumo as
datas que vieram erradas e converto a coluna `ph` (que tava em texto livre) pra número.


In [ ]:
df = pd.read_excel(
    'dadosbalneabilidade_tst.xlsx', 
    sheet_name='Laudo Antigo'
    )

### Checando dados e informações das colunas

In [ ]:
df.head()

In [ ]:
df.info()

* As colunas id, coluna de referencia a banco de dados, colunas não tem relevancia como as de tipo coleano.
* Retirando colunas que fazem referencia pessoais.

foi reduzido de 122 colunas para 46 colunas para inicio da analise.

In [ ]:
colunas = list(df.columns)

texto_comum = "tipo"

colunas_para_remover = [
    tipo for tipo in colunas if texto_comum.lower() in tipo.lower()
]

colunas_extras = [ 
    'numero',
    'responsavel', 
    'concluido',
    'id',
    'processo_id',
    'cliente_id', 
    'usuario_id',
    'processo_id2',
    'emitido',
    'impresso',
    'codigo_instituicao',
    'x',
    'y',
    'latitude',
    'longitude',
    'demanda',
    'hora_entra_laboratorio',
    'legenda_padrao',
    'codigo_hidro',
    'nome',
    'amostra'
    ]

for coluna in colunas_extras:
    colunas_para_remover.append(coluna)
    

df_intermediario = df.drop(columns=colunas_para_remover)

df_intermediario.info()

### Conversão de tipos e limpeza de datas/pH

### Verificando erros de digitação nas datas e corrigindo

Foi encontrado data ilegiveis, pode ter sido digitação ou erro de sistema.

'0201-05-26 00:00:00' 



In [ ]:
colunas_datetime = ['coleta', 'entrada' ,'data_conclusao']

for coluna in colunas_datetime:
    # Criamos uma série temporária para testar a conversão
    teste_conversao = pd.to_datetime(df_intermediario[coluna], errors='coerce', format='mixed')
    
    # Se houver valores que viraram NaT (e que não eram nulos antes), encontramos o erro
    erros_para_remover = df_intermediario[df_intermediario[coluna].notna() & teste_conversao.isna()].index
        
    if not erros_para_remover.empty:
        print(f"Erro encontrado na coluna: {coluna}")
        print("Valores Removidos !")
        #print(erros_para_remover[coluna].unique())
        df_intermediario.drop(erros_para_remover, inplace=True)
        

In [ ]:
colunas_datetime = ['coleta', 'entrada' ,'data_conclusao']

for coluna_datetime in colunas_datetime:
    
    df_intermediario[coluna_datetime] = pd.to_datetime(df_intermediario[coluna_datetime])
    

### Limpando coluna PH e convertendo para Float.

In [ ]:
# Supondo que sua coluna se chame 'coluna_problematica'
# 1. Tratando o "Zero": Convertendo variações de texto para o numeral 0
df_intermediario['ph_limpo'] = df_intermediario['ph'].astype(str).str.strip()

# Usamos regex para pegar 'Zero', 'zero', 'ZERO' e substituir por 0
df_intermediario['ph_limpo'] = df_intermediario['ph_limpo'].replace(r'(?i)^zero$', '0', regex=True)

# 2. Tratando o "<18" e "< 18": Removendo o símbolo de menor 
# e mantendo apenas o número 18 (ou o valor que você decidir para esse limite)
df_intermediario['ph_limpo'] = df_intermediario['ph_limpo'].str.replace(r'<\s*18', '18', regex=True)

# Substituímos '&#8805;' e '>' por nada, mantendo apenas o número
df_intermediario['ph_limpo'] = df_intermediario['ph_limpo'].astype(str).str.replace('&#8805;', '', regex=False)
df_intermediario['ph_limpo'] = df_intermediario['ph_limpo'].str.replace('>', '', regex=False)

# 1. Removendo o símbolo '<' e espaços extras
df_intermediario['ph_limpo'] = df_intermediario['ph_limpo'].astype(str).str.replace('<', '', regex=False).str.strip()

# Isso resolve os casos '<1,8' e '<0,001'
df_intermediario['ph_limpo'] = df_intermediario['ph_limpo'].str.replace(',', '.', regex=False)

# 2. Remoção de espaços em branco que possam ter sobrado
df_intermediario['ph_limpo'] = df_intermediario['ph_limpo'].str.strip()

df_intermediario['ph_limpo'] = pd.to_numeric(df_intermediario['ph_limpo'], errors='coerce')


In [ ]:
conversao = pd.to_numeric(df_intermediario['ph_limpo'], errors='coerce')

mascara_erros = conversao.isna() & df_intermediario['ph_limpo'].notna()

# 3. Contabilizamos a frequência de cada valor problemático
contagem_erros = df_intermediario.loc[mascara_erros, 'ph_limpo'].value_counts()

print("Ranking de erros (Valor : Quantidade):")
print(contagem_erros)

# 4. Se quiser ver o total absoluto de erros:
print(f"\nTotal de linhas com erro: {mascara_erros.sum()}")

### Analisando de Colunas Categoricas

Verificar se exite colunas categoricas, verificando a quantidade de valores únicos por coluna.

In [ ]:
print(df_intermediario.select_dtypes(include='object').nunique())

### Percentual de nulos no dataset já limpo

In [ ]:
# Seu cálculo
null_counts = (df_intermediario.isnull().sum() / len(df_intermediario) * 100).sort_values(ascending=False)

plt.figure(figsize=(10, 12)) # Ajustei a altura para as barras não ficarem espremidas
ax = sns.barplot(x=null_counts.values, y=null_counts.index, palette='coolwarm')

# Percorrer cada container de barras
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f%%', padding=5)

plt.title('Porcentagem de Dados Nulos por Coluna')
plt.xlabel('% de Nulos')
plt.ylabel('Colunas')
plt.xlim(0, 110) # Margem para o texto não sumir à direita
plt.show()

## 3. Enriquecimento com dados do INMET — aqui que eu achei o problema

### 3.1 O que eu tava fazendo de errado

O `inmet_filtrado.csv` não tem uma linha por dia. Ele tem, em média, umas **13 leituras por
dia** (é medição horária), e em alguns dias chega a 24. Só que eu juntava esse arquivo com os
laudos batendo só pela data:

```python
df_inmet_cleaned = df_inmet.dropna()
df_consolidado = pd.merge(df_intermediario, df_inmet_cleaned, on='entrada', how='left')
```

Juntar por data sem agregar antes é, na prática, multiplicar cada laudo pelo número de leituras
de clima daquele dia. O diagnóstico abaixo reproduz isso separado (sem mexer no
`df_intermediario` de verdade), só pra eu mostrar o tamanho do estrago.


In [ ]:
df_inmet_diagnostico = pd.read_csv('inmet_filtrado.csv')
df_inmet_diagnostico.rename(columns={'DATA': 'entrada'}, inplace=True)
df_inmet_diagnostico['entrada'] = pd.to_datetime(df_inmet_diagnostico['entrada'])

leituras_por_dia = df_inmet_diagnostico.groupby('entrada').size()
print("Leituras de INMET por dia — resumo:")
print(leituras_por_dia.describe())

_merge_diagnostico = pd.merge(
    df_intermediario, df_inmet_diagnostico.dropna(), on='entrada', how='left'
)
print(f"\nSHAPE df_intermediario (real):        {df_intermediario.shape}")
print(f"SHAPE apos merge SEM agregar (bug):    {_merge_diagnostico.shape}")
print(f"Fator de multiplicacao de linhas:      {_merge_diagnostico.shape[0] / df_intermediario.shape[0]:.2f}x")
del _merge_diagnostico


### 3.2 Por que isso é vazamento, não só "mais dado"

As cópias que o merge criava eram do mesmo laudo — mesmas variáveis biológicas, mesmo
`dentro` — só variando a hora da leitura de temperatura e radiação. Quando eu rodava o
`train_test_split` em cima disso, inevitavelmente algumas dessas cópias caíam no treino e
outras do mesmo laudo original caíam no teste. Ou seja, uma parte do que eu chamava de "teste"
já tinha sido visto, quase idêntico, no treino. Isso infla a acurácia sem eu perceber e joga
fora qualquer comparação séria entre fases ou modelos.

### 3.3 O que eu corrigi

Agreguei o INMET por dia (média de `TEMP_AR` e `RADIACAO`) antes de juntar com os laudos. Assim
cada dia vira uma linha só de clima, do mesmo jeito que `entrada` já é uma linha por dia nos
laudos.


In [ ]:
df_inmet = pd.read_csv('inmet_filtrado.csv')
df_inmet.rename(columns={'DATA': 'entrada'}, inplace=True)
df_inmet['entrada'] = pd.to_datetime(df_inmet['entrada'])

df_inmet_cleaned = df_inmet.dropna()

# CORREÇÃO: agregar por dia antes do merge (ver diagnóstico acima)
print('Linhas do INMET antes de agregar por dia:', df_inmet_cleaned.shape[0])
df_inmet_cleaned = df_inmet_cleaned.groupby('entrada', as_index=False)[['TEMP_AR', 'RADIACAO']].mean()
print('Linhas do INMET depois de agregar por dia:', df_inmet_cleaned.shape[0])
df_inmet_cleaned.info()

In [ ]:
# 1. Verifique os nomes das colunas (ajuste 'temperatura' para o nome real se necessário)
col_temp_1 = 'Temperatura' # Nome da coluna no df_intermediario
col_temp_2 = 'TEMP_AR' # Nome da coluna no df_inmet_cleaned

# 2. Fazendo o merge
df_consolidado = pd.merge(
    df_intermediario, 
    df_inmet_cleaned, 
    on='entrada', 
    how='left', 
    suffixes=('_original', '_extra')
)

# Se o Pandas não criou 'temperatura_original', ele manteve apenas 'temperatura'
nome_col_final_1 = col_temp_1 + '_original' if col_temp_1 + '_original' in df_consolidado.columns else col_temp_1
nome_col_final_2 = col_temp_2 + '_extra' if col_temp_2 + '_extra' in df_consolidado.columns else col_temp_2

# Preencher os nulos usando os nomes identificados
df_consolidado['temperatura_final'] = df_consolidado[nome_col_final_1].combine_first(df_consolidado[nome_col_final_2])

# Convertendo para FLoat
df_consolidado['temperatura_final'] = pd.to_numeric(df_consolidado['temperatura_final'], errors='coerce')
print("SHAPE df_intermediario:", df_intermediario.shape)
print("SHAPE df_consolidado apos merge corrigido:", df_consolidado.shape)
assert df_consolidado.shape[0] == df_intermediario.shape[0], "Merge ainda esta duplicando linhas!"
print("Colunas resultantes no merge:", df_consolidado.columns.tolist())

In [ ]:
df_consolidado.tail()

In [ ]:
# Conta quantas vezes a tag de abertura aparece
df_consolidado['contagem'] = df_consolidado['conclusao'].str.count('<b>')


# Criar a classificação
df_consolidado['tem_tag'] = df_consolidado['contagem'].apply(lambda x: 'Com Tag <b>' if x > 0 else 'Sem Tag')

# Contar os valores e plotar
df_consolidado['tem_tag'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'], rot=0)

plt.title('Comparação de Linhas: Presença de Tags <b>')
plt.ylabel('Quantidade de Linhas')
plt.show()

In [ ]:
# Seu cálculo
null_counts = (df_consolidado.isnull().sum() / len(df_consolidado) * 100).sort_values(ascending=False)

plt.figure(figsize=(10, 12)) # Ajustei a altura para as barras não ficarem espremidas
ax = sns.barplot(x=null_counts.values, y=null_counts.index, palette='coolwarm')

# Percorrer cada container de barras
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f%%', padding=5)

plt.title('Porcentagem de Dados Nulos por Coluna')
plt.xlabel('% de Nulos')
plt.ylabel('Colunas')
plt.xlim(0, 110) # Margem para o texto não sumir à direita
plt.show()

## 4. Analisando ruido das colunas de acordo com a resolução do Conama 274

* Turbidez
* Cor
* Nitrato
* Nitrogênio Amoniacal Total
* Coliformes Termotolerantes

#### Copia do Dataset

In [ ]:
df_conama = df_consolidado.copy()

#### Função para localizar padrões incorretos com Regex

In [ ]:
def localizar_erros_regex(df, coluna, padrao_correto):
    """
    df: seu DataFrame
    coluna: nome da coluna para investigar
    padrao_correto: a expressão regular do que DEVERIA estar lá
    """
    # Filtra as linhas onde o padrão NÃO é encontrado (ou seja, estão erradas)
    # O sinal de ~ inverte a máscara booleana
    erros = df[~df[coluna].astype(str).str.contains(padrao_correto, regex=True, na=False)]
    
    print(f"Total de registros fora do padrão na coluna '{coluna}': {len(erros)}")
    return erros[coluna].unique() # Retorna os valores únicos errados para facilitar a análise


#### Localizando Ruidos



Mostrando e fazendo uma analise de quais são os residos, olhando seu valores unicos.

Visto que em alguns residos pode transforma-los em valores categoricos.

In [ ]:
colunas_analise = ['Turbidez', 'Cor', 'Nitrato', 'Nitrogênio Amoniacal Total', 'Coliformes Termotolerantes']

for coluna in colunas_analise:
    erros = localizar_erros_regex(df=df_conama, coluna=coluna, padrao_correto=r'^\d+$')

    lista_filtrada = list(filter(lambda x: isinstance(x, (str, datetime)), erros))

    print(lista_filtrada)

### Quantificando ruidos

In [ ]:
print(df_conama[colunas_analise].isnull().sum() / len(df_conama) * 100)

### Criamos uma matriz booleana para visualizar o Ruido: 


True se o valor estiver na nossa lista de ruído.

* Zero
* Presença
* Ausência

In [ ]:
def plot_heatmap_ruido(df, colunas):
    
    ruido_total = ['Zero', 'ZERO', 'zero', 'Ausente', 'ausente', 'Ausência', 
                   'AUSENTE', 'Presença', 'PRESENTE', 'Presente', 'Presentes']
    
    # Criar DataFrame apenas com True/False para a presença de ruído
    df_ruido = df[colunas].isin(ruido_total)
    
    plt.figure(figsize=(15, 8))
    sns.heatmap(df_ruido, cbar=False, yticklabels=False, cmap='viridis')
    plt.title('Distribuição de Termos Qualitativos (Ruído) no Dataset')
    plt.xlabel('Variáveis de Qualidade da Água')
    plt.ylabel('Amostras (Linhas)')
    plt.show()

# Executar para as tuas colunas
plot_heatmap_ruido(df_conama, colunas_analise)

# 5. Metodo Fuzzy Matching controlado


## Fluxo:

1. Normaliza texto
2. Tenta match exato
3. Se não achar → tenta fuzzy
4. Se passar no threshold → mapeia
5. Se não passar → registra como desconhecido
6. Nunca cria coluna fora do padrão oficial

## Construção features sobre a coluna Conclusão.

Sugestão do professor: Outra analise a ser feita sobre a coluna conclusão, ela tem um texto que revela se está dentro de acordo com os padrões e se tem algo impedimento para não estar de acordo relata entre as tags "\<b\>" "\<\b\>"

In [ ]:
# Criando um novo DataFrame apenas com as colunas desejadas
df_conama_analise_conclusao = df_conama[['conclusao', 'dentro', 'RADIACAO', 'TEMP_AR', 'ph_limpo']].copy()

### Verificando valores nulos antes de começar

In [ ]:
# Seu cálculo
null_counts = (df_conama_analise_conclusao.isnull().sum() / len(df_conama_analise_conclusao) * 100).sort_values(ascending=False)

plt.figure(figsize=(10, 6)) # Ajustei a altura para as barras não ficarem espremidas
ax = sns.barplot(x=null_counts.values, y=null_counts.index, palette='coolwarm')

# Percorrer cada container de barras
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f%%', padding=5)

plt.title('Porcentagem de Dados Nulos por Coluna')
plt.xlabel('% de Nulos')
plt.ylabel('Colunas')
plt.xlim(0, 110) # Margem para o texto não sumir à direita
plt.show()

### Versão com fuzzy matching

In [ ]:
import re
import unicodedata
import pandas as pd
from rapidfuzz import process, fuzz

# ==========================================
# 1️⃣ NORMALIZAÇÃO PADRÃO CORPORATIVA
# ==========================================

def normalizar_texto(txt: str) -> str:
    if not txt:
        return ""

    txt = str(txt)

    txt = txt.replace('_x0001_', '').replace('&#7869;', 'e').replace('&amp;', 'e')

    txt = ''.join(
        c for c in unicodedata.normalize('NFD', txt)
        if unicodedata.category(c) != 'Mn'
    )

    txt = txt.lower()

    txt = re.sub(r'[^\w\s]', ' ', txt)
    txt = re.sub(r'\b(\w+)( \1\b)+', r'\1', txt)
    txt = re.sub(r'\s+', ' ', txt).strip()

    return txt


# ==========================================
# 2️⃣ DICIONÁRIO OFICIAL
# ==========================================

MAPEAMENTO_OFICIAL = {
    'amonia': 'Amônia Total',
    'amonia total': 'Amônia Total',
    'nitrogenio amoniacal total': 'Amônia Total',
    'nitrogenio amoniacal': 'Amônia Total',
    'clorofila': 'Clorofila "a"',
    'clorofila a': 'Clorofila "a"',
    'coliformes termotolerantes': 'Coliformes Termotolerantes',
    'coliformes totais': 'Coliformes Totais',
    'coliformes fecais': 'Coliformes Termotolerantes',
    'coliformes fecais escherichia coli': 'Coliformes Termotolerantes',
	'coliformes termotolerantes escherichia coli': 'Coliformes Termotolerantes',
    'enterococos': 'Coliformes Termotolerantes',
    'ecoli': 'Escherichia coli',
    'escherichia coli': 'Escherichia coli',
    'substancias soluveis em hexano': 'Substâncias Solúveis em Hexano',
    'soluveis em hexano': 'Substâncias Solúveis em Hexano',
    'solidos em suspensao': 'Sólidos em Suspensão',
    'solidos totais': 'Sólidos Totais',
    'solidos totais dissolvidos': 'Sólidos Totais Dissolvidos',
    'fosforo': 'Fósforo Total',
    'fosforo total': 'Fósforo Total',
    'cloretos fosforo total': 'Fósforo Total',
    'cloretos sodio': 'Cloretos Sodio',
    'dbo': 'DBO',
    'dqo': 'DQO',
    'od': 'Oxigênio Dissolvido',
    'oxigenio dissolvido': 'Oxigênio Dissolvido',
    'ph': 'pH',
    'cor': 'Cor',
    'cloretos': 'Cloretos',
    'oleos': 'Óleos e Graxas',
    'graxas': 'Óleos e Graxas',
    'nitrato': 'Nitrato',
    'nitrito': 'Nitrito',
    'turbidez': 'Turbidez',
	'cloro': 'Cloro',
	'cloro residual livre': 'Cloro',
	'cobre': 'Cobre',
	'condutividade': 'Condutividade Eletrica',
	'condutividade eletrica': 'condutividade eletrica',
	'condutuvidade eletrica': 'condutividade eletrica',
    'temperatura': 'Temperatura'
}

CHAVES_OFICIAIS = list(MAPEAMENTO_OFICIAL.keys())
NOMES_OFICIAIS = set(MAPEAMENTO_OFICIAL.values())

# ==========================================
# 3️⃣ FUZZY MATCH ENTERPRISE
# ==========================================

def fuzzy_match(chave, threshold=90):
    """
    Retorna nome oficial se similaridade >= threshold.
    """
    match = process.extractOne(
        chave,
        CHAVES_OFICIAIS,
        scorer=fuzz.token_sort_ratio
    )

    if match and match[1] >= threshold:
        chave_match = match[0]
        return MAPEAMENTO_OFICIAL[chave_match]

    return None


# ==========================================
# 4️⃣ EXTRAÇÃO ENTERPRISE
# ==========================================

def extrair_parametros_enterprise(texto, threshold=90, log_desconhecidos=None):

    tags = re.findall(r'<b>\s*(.*?)\s*</b>', str(texto))
    parametros = set()

    for bloco in tags:

        bloco = re.sub(r'\s+e\s+', ',', bloco)
        partes = bloco.split(',')

        for parte in partes:
            chave = normalizar_texto(parte)

            if not chave:
                continue

            # Match exato
            if chave in MAPEAMENTO_OFICIAL:
                parametros.add(MAPEAMENTO_OFICIAL[chave])
                continue

            # Fuzzy match
            nome_fuzzy = fuzzy_match(chave, threshold)

            if nome_fuzzy:
                parametros.add(nome_fuzzy)
            else:
                if log_desconhecidos is not None:
                    log_desconhecidos.add(chave)

    return list(parametros)


# ==========================================
# 5️⃣ EXECUÇÃO ESCALÁVEL
# ==========================================

log_desconhecidos = set()

df_conama_analise_conclusao['lista_parametros'] = (
    df_conama_analise_conclusao['conclusao']
    .apply(lambda x: extrair_parametros_enterprise(
        x,
        threshold=88,  # ajuste fino aqui
        log_desconhecidos=log_desconhecidos
    ))
)

# One-hot encoding robusto
df_explodido = df_conama_analise_conclusao['lista_parametros'].explode()

df_parametros = pd.crosstab(
    df_explodido.index,
    df_explodido
)

df_conama_analise_conclusao_final = df_conama_analise_conclusao.join(df_parametros)
df_conama_analise_conclusao_final = df_conama_analise_conclusao_final.fillna(0)

# Garante inteiro apenas nas colunas novas
for col in df_parametros.columns:
    df_conama_analise_conclusao_final[col] = df_conama_analise_conclusao_final[col].astype(int)


# ==========================================
# 6️⃣ LOG DE GOVERNANÇA
# ==========================================

df_termos_nao_mapeados = pd.DataFrame({
    "termo_nao_mapeado": sorted(log_desconhecidos)
})



In [ ]:
df_conama_analise_conclusao_final.head()

### Ranking dos Parâmetros Mais Frequentes

Rodando isso, você saberá exatamente qual elemento químico é o "vilão" número 1 da sua base de dados:

In [ ]:
# Definir a lista de colunas que devem ser ignoradas
colunas_para_ignorar = ['conclusao', 'dentro', 'lista_parametros', 'RADIACAO', 'TEMP_AR', 'ph_limpo', 'conclusao_caracteres']

# Filtrar apenas colunas numéricas E que NÃO estão na lista de ignoradas
colunas_parametros = [
    col for col in df_conama_analise_conclusao_final.select_dtypes(include=['number']).columns 
    if col not in colunas_para_ignorar
]

# Calcular Frequência Total
frequencia_total = df_conama_analise_conclusao_final[colunas_parametros].sum().sort_values(ascending=False).reset_index()
frequencia_total.columns = ['Parâmetro', 'Total de Ocorrências']

# Ranking Geral de Irregularidades
plt.figure(figsize=(12, 10))
sns.set_style("whitegrid")
ax = sns.barplot(
    x='Total de Ocorrências', 
    y='Parâmetro', 
    data=frequencia_total, 
    palette='magma'
)

plt.title('Ranking Geral de Parâmetros Fora do Padrão (Filtrado)', fontsize=16)
plt.xlabel('Número de Amostras Irregulares', fontsize=12)
plt.ylabel('Parâmetros Analisados', fontsize=12)

for i in ax.containers:
    ax.bar_label(i, padding=3)

plt.tight_layout()
plt.show()

### Gráficos de Frequência e Correlação para as colunas ['Turbidez', 'Cor', 'Nitrato', 'Amônia Total', 'Coliformes Termotolerantes']


* Permite uma comparação direta. 

Ex.: Se ele mostrará a barra de Coliformes Termotolerantes muito maior que as outras, destacando a urgência de intervenção biológica.

* É excelente para a análise categórica. 

Ex.: Se houver um valor alto (próximo de 1.0) entre "Cor" e "Turbidez", por exemplo, você sabe que tratar a cor provavelmente resolverá a turbidez simultaneamente.

In [ ]:
# Configuração visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Preparando os dados de frequência (baseado no ranking anterior)
colunas_analise = ['Turbidez', 'Cor', 'Nitrato', 'Amônia Total', 'Coliformes Termotolerantes']
frequencia = df_conama_analise_conclusao_final[colunas_analise].sum().sort_values(ascending=False).reset_index()
frequencia.columns = ['Parâmetro', 'Total de Irregularidades']

# Frequência de Irregularidades
plt.figure(figsize=(10, 6))
ax = sns.barplot(x='Total de Irregularidades', y='Parâmetro', data=frequencia, palette='viridis')
plt.title('Frequência de Parâmetros Fora do Padrão', fontsize=15)
plt.xlabel('Número de Amostras Irregulares')
plt.ylabel('Parâmetro Analisado')

# Adicionando os rótulos de dados nas barras
for i in ax.containers:
    ax.bar_label(i, padding=3)

plt.tight_layout()
plt.show()

# Heatmap de Correlação (Ocorrências Simultâneas)
# Mostra se quando o parâmetro A falha, o B também costuma falhar
plt.figure(figsize=(8, 6))
sns.heatmap(df_conama_analise_conclusao_final[colunas_analise].corr(), annot=True, cmap='Reds', fmt='.2f')
plt.title('Correlação de Falhas entre Parâmetros', fontsize=15)
plt.tight_layout()
plt.show()

### Nuvem de Elementos quimicos

Destaca instantaneamente quais parâmetros aparecem com mais frequência nas descrições de irregularidade.

In [ ]:
from wordcloud import WordCloud

# 1. Preparar o texto
# Convertemos a coluna para string, removemos colchetes e aspas para limpar os nomes
texto_parametros = " ".join(df_conama_analise_conclusao_final['lista_parametros'].astype(str))
limpezas = ["[", "]", "'", ",", "None"]
for char in limpezas:
    texto_parametros = texto_parametros.replace(char, "")

# 2. Configurar a Nuvem de Palavras
nuvem = WordCloud(
    width=1000, 
    height=600, 
    background_color='white',
    colormap='Reds',      # Tons de vermelho para indicar criticidade
    min_font_size=10,
    max_words=100
).generate(texto_parametros)

# 3. Exibir o gráfico
plt.figure(figsize=(12, 8))
plt.imshow(nuvem, interpolation='bilinear')
plt.axis("off")  # Remove os eixos do gráfico
plt.title("Nuvem de Parâmetros Fora dos Padrões", fontsize=20, pad=20)
plt.tight_layout(pad=0)
plt.show()

### Contagem de caracteres da coluna conclusão

In [ ]:
df_conama_analise_conclusao_final['conclusao_caracteres'] = df_conama_analise_conclusao_final['conclusao'].astype(str).str.len()

## 6. Seleção de variáveis e prevalência

Daqui pra frente, todo número que aparece já é da base corrigida — uma linha por laudo, sem
duplicação de leitura de clima.


### Gráfico de Barras Empilhadas (Percentual)



Gráfico de barras horizontais onde cada barra representa uma variável e o comprimento mostra a proporção de **True** vs. **False**.

Vantagem: Permite comparar 20 ou 30 variáveis de uma só vez.

Ordene as barras pela frequência de "Verdadeiro" para identificar rapidamente quais comportamentos são dominantes.

1. O que os eixos representam

    * Eixo Vertical (Y): Lista todas as suas variáveis booleanas (os nomes dos parâmetros químicos e biológicos).

    * Eixo Horizontal (X): Mostra a proporção de ocorrência. Os valores vão de $0.00$ a $1.00$ (ou $0\%$ a $100\%$). No seu gráfico, o valor máximo é cerca de $0.20$ ($20\%$).

2. Leitura dos Dados

    O gráfico está ordenado da variável mais frequente para a menos frequente, o que facilita muito a análise:

    * Líderes de Presença: "Fósforo Total" é a variável que mais aparece como True no seu dataset (em aproximadamente $20\%$ das amostras), seguida por "Cor" e "Coliformes Termotolerantes".
    * Ocorrências Raras: Variáveis como "Óleos e Graxas" e "Cloro" aparecem em menos de $1\%$ dos casos.
    * Variáveis "Zeradas": Parâmetros como "Temperatura", "Cobre" e "Sólidos Totais" não possuem nenhuma barra. Isso significa que, em todo o seu dataset, elas são sempre False (ou $0$).

3. Insights Práticos

    * Sparsity (Esparsidade): Seu dataset é altamente esparso. Mesmo a variável mais comum só aparece em $20\%$ das vezes. Isso sugere que a maioria das suas linhas tem muitos valores False.
    * Foco de Análise: Se você for fazer um modelo de previsão ou correlação, deve focar nas top 10 variáveis. As que estão no final da lista (as zeradas) não trazem informação estatística útil para modelos de machine learning, pois não têm variância.
    * Redundância: Note que existem "Coliformes Termotolerantes" com alta frequência e "Coliformes Totais" com zero. Isso pode indicar uma diferença na forma como os dados foram coletados ou uma regra de negócio específica.

In [ ]:
# Supondo que seja seu dataframe com booleanas
# 1. Transformar em numérico e calcular a média (proporção de True)
df_proporcao = df_conama_analise_conclusao_final.drop(
    columns=[
        'conclusao', 
        'dentro', 
        'lista_parametros',
        'ph_limpo', 
        'RADIACAO', 
        'conclusao_caracteres',
        'TEMP_AR']
    )
proporcao = df_proporcao.mean().sort_values(ascending=False).reset_index()

# Ordenando e preparando os dados
proporcao.columns = ['Variavel', 'Proporcao']

# Configurando o estilo
sns.set_style("whitegrid")
plt.figure(figsize=(10, 8))

# Criando o gráfico com um degradê baseado no valor
pal = sns.color_palette("viridis_r", len(proporcao))
ax = sns.barplot(x='Proporcao', y='Variavel', data=proporcao, palette=pal)

# Adicionando os valores nas barras para facilitar a leitura
for p in ax.patches:
    ax.annotate(f'{p.get_width()*100:.1f}%', 
                (p.get_width(), p.get_y() + p.get_height() / 2),
                xytext=(5, 0), textcoords='offset points', 
                va='center', fontsize=10, fontweight='bold')

plt.title('Prevalência de Parâmetros Fora do Conformidade (%)', fontsize=14, pad=20)
plt.xlabel('Frequência Relativa (Presença)', fontsize=12)
plt.ylabel('')
plt.xlim(0, proporcao['Proporcao'].max() * 1.15) # Dá espaço para o texto
sns.despine(left=True, bottom=True)

plt.show()

* Remoção de "Constantes": O código remove automaticamente variáveis como Cobre ou Sólidos Totais que estavam zeradas no seu gráfico. Em estatística, variáveis sem variância não ajudam o modelo a aprender.

* Foco em Nutrientes e Matéria Orgânica: Ao filtrar pelo threshold, o código manterá Fósforo Total, Amônia, DBO e Turbidez. Esses são os "impulsionadores" do crescimento bacteriano.

* Redução de Dimensionalidade: Ao focar nas top variáveis, você evita o overfitting (quando o modelo decora o ruído em vez de aprender o padrão).

In [ ]:
# 1. Definir as variáveis "Obrigatórias" (Indicadores Biológicos/Padrão)
obrigatorias = [
    'Coliformes Termotolerantes',     
    'Enterococos'
]

# 2. Filtragem Automática de Variáveis de Contexto
# Vamos manter apenas variáveis que possuem variabilidade (não são sempre False)
# e que aparecem em pelo menos 1% das amostras para evitar ruído.
threshold = 0.01
frequencias = df_proporcao.mean()

contexto_relevante = frequencias[
    (frequencias >= threshold) & 
    (~frequencias.index.isin(obrigatorias))
].sort_values(ascending=False).index.tolist()

# 3. Criar o Dataset Final de Modelagem
# Unimos as obrigatórias que existem no seu DF com as de contexto filtradas
colunas_selecionadas = [c for c in obrigatorias if c in df_proporcao.columns] + contexto_relevante
df_modelagem = df_proporcao[colunas_selecionadas]

print(f"Variáveis selecionadas para o modelo ({len(colunas_selecionadas)}):")
print(colunas_selecionadas)

In [ ]:
print("Variaveis retidas para modelagem:", list(df_modelagem.columns))
print(f"Total de variaveis retidas: {len(df_modelagem.columns)}")


In [ ]:
proporcao = df_modelagem.mean().sort_values(ascending=False).reset_index()

# Ordenando e preparando os dados
proporcao.columns = ['Variavel', 'Proporcao']

# Configurando o estilo
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))

# Criando o gráfico com um degradê baseado no valor
pal = sns.color_palette("viridis_r", len(proporcao))
ax = sns.barplot(x='Proporcao', y='Variavel', data=proporcao, palette=pal)

# Adicionando os valores nas barras para facilitar a leitura
for p in ax.patches:
    ax.annotate(f'{p.get_width()*100:.1f}%', 
                (p.get_width(), p.get_y() + p.get_height() / 2),
                xytext=(5, 0), textcoords='offset points', 
                va='center', fontsize=10, fontweight='bold')

plt.title('Prevalência de Parâmetros Fora do Conformidade (%)', fontsize=14, pad=20)
plt.xlabel('Frequência Relativa (Presença)', fontsize=12)
plt.ylabel('')
plt.xlim(0, proporcao['Proporcao'].max() * 1.15) # Dá espaço para o texto
sns.despine(left=True, bottom=True)

plt.show()

In [ ]:
print("Prevalencia de nao-conformidade por parametro (dados corrigidos):")
print((proporcao.set_index('Variavel')['Proporcao'] * 100).round(1).astype(str) + '%')


# 7. Analise da distribuição de probabilidade em relação  da features/variavel com o tarquet/alvo.

## Análisando Coliformes Termotolerantes


### Conclusão Coliformes Termotolerantes


A análise mostra que os Coliformes Termotolerantes são um excelente filtro para descartar praias ruins, mas são insuficientes para confirmar praias boas.

Os casos "Impróprios" que ocorrem mesmo com Coliformes em zero (a parte vermelha na coluna 0) são causados pelos outros parâmetros que vimos no primeiro gráfico, como Fósforo Total, Cor ou Amônia.

### Gráfico de Densidade com Rug Plot

1. O que os Eixos e as Cores Significam

    * Eixo X (0 e 1): Representa sua variável booleana. 0 significa que Coliformes não foram detectados; 1 significa que foram detectados.
    * Curva Verde (Própria - 1.0): Representa a distribuição das amostras de água que passaram no teste de balneabilidade.
    * Curva Laranja (Imprópria - 0.0): Representa as amostras que falharam no teste.
    * Eixo Y (Densidade): Indica onde a maior parte dos dados está concentrada. Quanto mais alta a "montanha", maior a probabilidade de encontrar aquele estado naquele ponto.

2. A "Montanha" Verde no Ponto Zero

    Note que há um pico verde altíssimo e muito estreito exatamente sobre o 0.

    * O que isso diz: Quase 100% das vezes em que a água é considerada Própria, os Coliformes estão em 0.
    * Conclusão para o modelo: A ausência de Coliformes é uma condição obrigatória (mas não única) para a água estar boa.

3. A Separação no Ponto Um

    Veja que no ponto 1.0, existe apenas uma curva laranja suave; a curva verde é inexistente ali.

    * O que isso diz: Se o valor de Coliformes pula para 1, a probabilidade de a água estar própria cai para virtualmente zero.
    * Conclusão para o modelo: Esta variável é um "Filtro de Exclusão". Se der 1, o modelo já sabe que a resposta é "Imprópria" sem precisar olhar mais nada.

4. O "Problema" no Ponto Zero (A Sobreposição)

    Observe a base do gráfico no ponto 0. Existe uma pequena "lombada" laranja sob a montanha verde.

    * O que isso diz: Existem casos onde os Coliformes são 0 (não detectados), mas a água ainda assim é Imprópria.
    * Por que isso acontece? Provavelmente porque outros parâmetros (como Fósforo, Cor ou Amônia, que vimos no seu primeiro gráfico) estão fora dos limites, mesmo com a parte bacteriológica estando "limpa".

In [ ]:
# Configuração de estilo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 7))

# 1. Gráfico de Densidade com Rug Plot (mostra onde cada dado individual está)
ax = sns.kdeplot(data=df_conama_analise_conclusao_final, 
                 x='Coliformes Termotolerantes', 
                 hue='dentro', 
                 fill=True, 
                 common_norm=False, 
                 palette='RdYlGn',
                 alpha=0.5,
                 linewidth=2)

# 2. Adicionar Rug Plot para ver a concentração real dos pontos nos eixos 0 e 1
sns.rugplot(data=df_conama_analise_conclusao_final, 
            x='Coliformes Termotolerantes', 
            hue='dentro', 
            palette='RdYlGn', 
            alpha=0.3)

# Customização de Legendas e Títulos
plt.title('Análise de Separação: Coliformes Termotolerantes vs Balneabilidade\n(Curvas de Densidade Probabilística)', 
          fontsize=15, pad=20)
plt.xlabel('Variável Booleana: Coliformes (0 = Não Detectado, 1 = Detectado)', fontsize=12)
plt.ylabel('Densidade', fontsize=12)

# Melhorar a legenda
legend = ax.get_legend()
legend.set_title("Status Balneabilidade")
for t, l in zip(legend.texts, ("Imprópria (0.0)", "Própria (1.0)")):
    t.set_text(l)

plt.tight_layout()
plt.show()


### Figura com dois subplots

Painel que combina a frequência com a probabilidade condicional. Isso vai te dar a "assinatura" exata de como essa variável dita o resultado.

1. Gráfico de Volume (Esquerda)

    Este gráfico de barras mostra a quantidade bruta de registros no seu dataset.

    * Ausência de Coliformes (0): É o cenário mais comum. Note que, quando não há coliformes (barra 0), a grande maioria das amostras está Própria (verde/dentro=1.0), mas ainda existe uma quantidade considerável de amostras Impróprias (laranja/dentro=0.0).
    * Presença de Coliformes (1): Quando coliformes são detectados, a barra verde simplesmente desaparece. Só existem registros laranjas.
    * Insight: Isso confirma que a presença de coliformes é um "atestado" de água imprópria, mas a ausência deles não garante que a água esteja própria.

2. Gráfico de Probabilidade (Direita)

    Este é o gráfico mais importante para a sua predição, pois ele isola a probabilidade.

    * Coluna 0 (Sem Coliformes): Se o resultado for negativo para coliformes, a chance de a água estar própria é de aproximadamente 60% (parte verde) contra 40% de estar imprópria (parte vermelha). Ou seja, sem coliformes, o resultado é incerto.
    * Coluna 1 (Com Coliformes): Se o resultado for positivo, a probabilidade de a água estar própria cai para 0%. A barra é 100% vermelha.
    * Insight: O modelo de Machine Learning aprenderá que:
    
    Se Coliformes == 1 -> 100% de chance de Imprópria    
    Se Coliformes == 0 -> Preciso de outras variáveis (Fósforo, Cor, etc) para decidir$$

In [ ]:
# Criando a figura com dois subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. Gráfico de Barras Empilhadas (Volume Total)
# Mostra quantos casos temos de cada combinação
sns.countplot(data=df_conama_analise_conclusao_final, x='Coliformes Termotolerantes', 
              hue='dentro', palette='RdYlGn', ax=ax1)
ax1.set_title('Volume de Amostras: Coliformes vs Balneabilidade')
ax1.set_ylabel('Quantidade de Registros')
legend = ax1.get_legend()
legend.set_title("Status Balneabilidade")
for t, l in zip(legend.texts, ("Imprópria (0.0)", "Própria (1.0)")):
    t.set_text(l)

# 2. Gráfico de Probabilidade (Eficiência do Preditor)
# Calcula a % de chance de estar própria/imprópria
prop_data = df_conama_analise_conclusao_final.groupby('Coliformes Termotolerantes')['dentro'].value_counts(normalize=True).unstack()
prop_data.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], ax=ax2)

ax2.set_title('Probabilidade: Chance de estar Própria (%)')
ax2.set_ylabel('Proporção (0 a 1.0)')
ax2.legend(title='Própria (1.0)', loc='upper right')
legend = ax2.get_legend()
legend.set_title("Status Balneabilidade")
for t, l in zip(legend.texts, ("Imprópria (0.0)", "Própria (1.0)")):
    t.set_text(l)

plt.tight_layout()
plt.show()

## Análisando Fósforo Total

### Conclusão Fósforo Total

Este gráfico é fundamental porque ele explica o que os **Coliformes** sozinhos não conseguiam: os casos de água imprópria que ocorrem mesmo sem bactérias detectadas.

Para entender o papel do **Fósforo Total** na sua predição de balneabilidade. Enquanto o gráfico anterior de Coliformes era radical, este mostra uma relação de **probabilidade acumulada**.

Se você observar bem, o comportamento do Fósforo é muito parecido com o dos Coliformes: **ambos são ótimos para prever quando a água está RUIM.**

**O insight para sua predição:**

Para o seu modelo de balneabilidade, você não precisa apenas de um indicador. Você está construindo uma lógica de "filtros":

1. Filtro 1: Tem Coliformes? Se sim $\rightarrow$ Imprópria.
2. Filtro 2: Tem Fósforo Total? Se sim $\rightarrow$ Imprópria.
3. Filtro 3: Tem Cor alta? Se sim $\rightarrow$ Imprópria.

A água só será considerada Própria se ela passar por todos esses filtros sem "ativar" nenhum deles.

### Gráfico de Densidade com Rug Plot

1. A Variável Fósforo como "Sinalizador de Esgoto"

    O Fósforo Total é frequentemente um indicador de carga orgânica (esgoto doméstico ou fertilizantes). No gráfico:

    * Pico Verde no Zero: Novamente, vemos que para a água estar Própria (1.0), o Fósforo quase sempre deve estar em 0 (não detectado ou abaixo do limite).
    * A "Lombada" Laranja no Um: Veja que existe uma concentração significativa de amostras Impróprias (0.0) quando o Fósforo Total é detectado (1.0).

2. Por que ele é diferente dos Coliformes?

    Se você comparar com o gráfico de Coliformes, notará que a "montanha" laranja no ponto 0.0 (água imprópria sem o parâmetro presente) é menor aqui do que no gráfico de Coliformes.

    * O que isso significa: O Fósforo Total consegue "capturar" e explicar alguns casos de poluição que os Coliformes deixam passar.
    * Complementaridade: Eles funcionam como uma rede. O que escapa do buraco da rede de Coliformes, a rede de Fósforo segura.

3. Conclusão para o Modelo de Predição

    Este gráfico confirma que o seu modelo de Machine Learning terá muito sucesso se combinar essas variáveis:

    * Coliformes Termotolerantes: É o seu preditor de "choque" (se aparecer, está impróprio).
    * Fósforo Total: É o seu preditor de "contexto" (ajuda a identificar águas contaminadas por nutrientes mesmo quando as bactérias ainda não proliferaram ou já morreram).

In [ ]:
# Configuração de estilo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 7))

# 1. Gráfico de Densidade com Rug Plot (mostra onde cada dado individual está)
ax = sns.kdeplot(data=df_conama_analise_conclusao_final, 
                 x='Fósforo Total', 
                 hue='dentro', 
                 fill=True, 
                 common_norm=False, 
                 palette='RdYlGn',
                 alpha=0.5,
                 linewidth=2)

# 2. Adicionar Rug Plot para ver a concentração real dos pontos nos eixos 0 e 1
sns.rugplot(data=df_conama_analise_conclusao_final, 
            x='Fósforo Total', 
            hue='dentro', 
            palette='RdYlGn', 
            alpha=0.3)

# Customização de Legendas e Títulos
plt.title('Análise de Separação: Fósforo Total vs Balneabilidade\n(Curvas de Densidade Probabilística)', 
          fontsize=15, pad=20)
plt.xlabel('Variável Booleana: Fósforo Total (0 = Não Detectado, 1 = Detectado)', fontsize=12)
plt.ylabel('Densidade', fontsize=12)

# Melhorar a legenda
legend = ax.get_legend()
legend.set_title("Status Balneabilidade")
for t, l in zip(legend.texts, ("Imprópria (0.0)", "Própria (1.0)")):
    t.set_text(l)

plt.tight_layout()
plt.show()


### Figura com dois subplots

1. Volume de Amostras (Esquerda)

    Este gráfico de barras conta quantos registros você tem para cada situação:

    * Barra 0 (Fósforo Ausente): A grande maioria das suas amostras não apresenta Fósforo Total acima do limite. Note que, quando não há Fósforo (0), a barra verde (Própria) é muito maior que a laranja.
    
    * Barra 1 (Fósforo Presente): Quando o Fósforo é detectado, a situação se inverte drasticamente. Quase não há amostras próprias (verde), e o volume de amostras Impróprias (laranja) torna-se dominante.

2. Chance de estar Própria (%) (Direita)

    Este gráfico é o que o seu modelo de inteligência artificial realmente vai "ler":

    * Na Coluna 0: Se o Fósforo é zero, a chance de a água estar própria é alta (cerca de 70%), mas ainda existe um risco de 30% de estar imprópria por outros motivos.

    * Na Coluna 1: Quando o Fósforo Total é detectado, a probabilidade de a água estar própria cai para quase zero. A barra é praticamente toda vermelha.

In [ ]:
# Criando a figura com dois subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de Barras Empilhadas (Volume Total)
# Mostra quantos casos temos de cada combinação
sns.countplot(data=df_conama_analise_conclusao_final, x='Fósforo Total', 
              hue='dentro', palette='RdYlGn', ax=ax1)
ax1.set_title('Volume de Amostras: Fósforo Total vs Balneabilidade')
ax1.set_ylabel('Quantidade de Registros')
legend = ax1.get_legend()
legend.set_title("Status Balneabilidade")
for t, l in zip(legend.texts, ("Imprópria (0.0)", "Própria (1.0)")):
    t.set_text(l)

# Gráfico de Probabilidade (Eficiência do Preditor)
# Calcula a % de chance de estar própria/imprópria
prop_data = df_conama_analise_conclusao_final.groupby('Fósforo Total')['dentro'].value_counts(normalize=True).unstack()
prop_data.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], ax=ax2)

ax2.set_title('Probabilidade: Chance de estar Própria (%)')
ax2.set_ylabel('Proporção (0 a 1.0)')
ax2.legend(title='Própria (1.0)', loc='upper right')
legend = ax2.get_legend()
legend.set_title("Status Balneabilidade")
for t, l in zip(legend.texts, ("Imprópria (0.0)", "Própria (1.0)")):
    t.set_text(l)

plt.tight_layout()
plt.show()

## Análisando Cor

### Gráfico de Densidade com Rug Plot

1. O Poder de Exclusão da "Cor"

    Observe as duas extremidades do gráfico (pontos 0.0 e 1.0):

    * No ponto 1.0 (Cor Detectada): Existe apenas a curva laranja (Imprópria). Não há praticamente nenhuma presença da curva verde (Própria).
        * Insight: Assim como os Coliformes, se a variável "Cor" for ativada (True/1), a água é automaticamente classificada como imprópria. Ela é um indicador de "alerta máximo".
    * No ponto 0.0 (Cor Ausente): Temos o pico verde altíssimo, indicando que a ausência de alteração na cor é um requisito para a água estar própria. No entanto, note a pequena área laranja na base do zero.
        * Insight: Água visualmente limpa (Cor = 0) não é garantia de balneabilidade, pois pode haver bactérias invisíveis a olho nu.

2. Comparação com Fósforo e Coliformes

    Ao observar este gráfico em conjunto com os anteriores, você pode notar uma hierarquia de "rigidez" nas variáveis:

    * Coliformes Termotolerantes: É a variável biológica direta.
    * Fósforo Total: É o indicador químico de carga orgânica.
    * Cor: É o indicador físico/estético.

3. Conclusão para o Modelo de Predição

    A variável Cor é excelente para o seu modelo porque ela tem baixa taxa de "Falsos Negativos" para impropriedade. Se a cor está alterada, a água está ruim. Ponto.

    **O que o seu modelo aprenderá com esses 3 gráficos combinados?**
    
    1. **Se Cor == 1 OU Coliformes == 1 OU Fósforo == 1 $\rightarrow$ Imprópria.**    
    2. **Se todos forem 0 $\rightarrow$ Alta probabilidade de estar Própria**, mas o modelo ainda precisará checar variáveis secundárias como Amônia ou DBO para ter certeza absoluta.

In [ ]:
# Configuração de estilo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 7))

# 1. Gráfico de Densidade com Rug Plot (mostra onde cada dado individual está)
ax = sns.kdeplot(data=df_conama_analise_conclusao_final, 
                 x='Cor', 
                 hue='dentro', 
                 fill=True, 
                 common_norm=False, 
                 palette='RdYlGn',
                 alpha=0.5,
                 linewidth=2)

# 2. Adicionar Rug Plot para ver a concentração real dos pontos nos eixos 0 e 1
sns.rugplot(data=df_conama_analise_conclusao_final, 
            x='Cor', 
            hue='dentro', 
            palette='RdYlGn', 
            alpha=0.3)

# Customização de Legendas e Títulos
plt.title('Análise de Separação: Cor vs Balneabilidade\n(Curvas de Densidade Probabilística)', 
          fontsize=15, pad=20)
plt.xlabel('Variável Booleana: Fósforo Total (0 = Não Detectado, 1 = Detectado)', fontsize=12)
plt.ylabel('Densidade', fontsize=12)

# Melhorar a legenda
legend = ax.get_legend()
legend.set_title("Status Balneabilidade")
for t, l in zip(legend.texts, ("Imprópria (0.0)", "Própria (1.0)")):
    t.set_text(l)

plt.tight_layout()
plt.show()


### Figura com dois subplots

1. Volume de Amostras (Gráfico da Esquerda)

    * Status "0" (Cor Normal): A grande maioria das amostras se concentra aqui. Quando a cor está normal, há uma predominância clara de águas Próprias (barra verde), mas ainda existe um volume significativo de águas Impróprias (barra laranja).

    * Status "1" (Cor Alterada): Note que a barra verde (Própria) praticamente desaparece. Quase a totalidade das amostras com cor alterada são classificadas como Impróprias.

2. Chance de estar Própria (Gráfico da Direita)

    Este gráfico de proporção é o "manual de instruções" para o seu algoritmo:

    * Quando Cor é 0: A probabilidade de a água estar própria é de aproximadamente 63%. Ou seja, "não ter cor" não é garantia de balneabilidade, mas é um bom começo.

    * Quando Cor é 1: A probabilidade de a água estar própria cai para perto de 0%. A barra é quase totalmente vermelha (Imprópria).

**Por que esse gráfico é o fechamento perfeito para sua análise?**

Ao olhar para os gráficos de Coliformes, Fósforo e agora Cor, você percebe que eles compartilham um padrão de "Probabilidade Unidirecional":

* Poder de Rejeição: Todas as três variáveis são excelentes para dizer quando a água está RUIM. Se qualquer uma delas for "1", a chance de estar própria morre.

* Incerteza no "Zero": O maior desafio do seu modelo de predição está em classificar as amostras onde todos esses indicadores são "0". É nesse cenário que o modelo precisará cruzar os dados (ex: se Fósforo é 0 E Cor é 0 E Coliformes é 0, a chance de estar própria sobe para o patamar de 80-90%).

**Resumo da sua Estratégia de Predição**

Você agora tem as 3 "âncoras" do seu modelo. Juntas, elas cobrem os três pilares da balneabilidade:

1. **Biológico:** Coliformes Termotolerantes.
2. **Químico:** Fósforo Total.
3. **Físico:** Cor.

In [ ]:
# Criando a figura com dois subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de Barras Empilhadas (Volume Total)
# Mostra quantos casos temos de cada combinação
sns.countplot(data=df_conama_analise_conclusao_final, x='Cor', 
              hue='dentro', palette='RdYlGn', ax=ax1)
ax1.set_title('Volume de Amostras: Cor vs Balneabilidade')
ax1.set_ylabel('Quantidade de Registros')
legend = ax1.get_legend()
legend.set_title("Status Balneabilidade")
for t, l in zip(legend.texts, ("Imprópria (0.0)", "Própria (1.0)")):
    t.set_text(l)

# Gráfico de Probabilidade (Eficiência do Preditor)
# Calcula a % de chance de estar própria/imprópria
prop_data = df_conama_analise_conclusao_final.groupby('Cor')['dentro'].value_counts(normalize=True).unstack()
prop_data.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], ax=ax2)

ax2.set_title('Probabilidade: Chance de estar Própria (%)')
ax2.set_ylabel('Proporção (0 a 1.0)')
ax2.legend(title='Própria (1.0)', loc='upper right')
legend = ax2.get_legend()
legend.set_title("Status Balneabilidade")
for t, l in zip(legend.texts, ("Imprópria (0.0)", "Própria (1.0)")):
    t.set_text(l)

plt.tight_layout()
plt.show()

## Mapa de Calor de Correlação (Heatmap)



Se você quer entender como essas variáveis se relacionam (ex: "Sempre que A é verdadeiro, B também é?"), o Heatmap é essencial.

Como fazer: Calcule a matriz de correlação (usando o coeficiente de Phi ou Spearman) entre todas as booleanas.

O que buscar: Blocos de cor intensa indicam variáveis que andam juntas.

In [ ]:
# 1. Preparação: Garante que os dados sejam numéricos (0 e 1)
# Se suas colunas forem True/False, o pandas converte automaticamente para 1/0
df_numeric = df_modelagem.astype(int)

# 2. Calcula a matriz de correlação
corr = df_numeric.corr()

# 3. Configura o visual do Heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(corr, 
            annot=True,      # Mostra os números dentro dos quadrados
            fmt=".2f",       # Duas casas decimais
            cmap='coolwarm', # Azul para correlação negativa, vermelho para positiva
            center=0,        # Garante que o branco seja o ponto neutro
            square=True)     # Deixa os campos perfeitamente quadrados

plt.title('Mapa de Calor: Correlação entre Variáveis Booleanas')
plt.show()

# 8. Treinando os modelos — as 3 fases de novo

Mesmo desenho da análise original: Random Forest, Gradient Boosting e XGBoost, em 3 fases de
variáveis, cada uma com até 3 níveis de complexidade. A diferença agora é que a base tá
correta: 11.087 laudos, split 80/20 com `random_state=42`, e o teste de verdade fica com
**2.218 amostras** — não as 19.372 que eu tinha antes (esse número, sozinho, já era um sinal de
que tinha algo errado).


### Fase 1 — Filtro Biológico Primário

Só as 3 variáveis mais importantes da balneabilidade: Coliformes Termotolerantes, Fósforo
Total e Cor.


**Simples (baseline)**

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 1. Preparação dos Dados (conforme sua análise)
features = ['Coliformes Termotolerantes', 'Fósforo Total', 'Cor']
X = df_conama_analise_conclusao_final[features].astype(int)
y = df_conama_analise_conclusao_final['dentro'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definição dos Modelos (Dicionário para facilitar o loop)
modelos = {
    "RF_Simples": RandomForestClassifier(random_state=42),
    "GB_Simples": GradientBoostingClassifier(random_state=42),
    "XGB_Simples": XGBClassifier(random_state=42),
}

# 3. Treinamento e Exibição de Resultados
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

resultados = {}

for i, (nome, modelo) in enumerate(modelos.items()):
    # Treino
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    resultados[nome] = acc
    
    print(f"\n{'='*30}\nMODELO: {nome}\n{'='*30}")
    print(classification_report(y_test, y_pred, target_names=['Imprópria (0)', 'Própria (1)']))
    
    # Matriz de Confusão Visual
    cm = confusion_matrix(y_test, y_pred)
    print(f"{nome:22s} | Acc: {acc:.4f} | Matriz (linhas=real, colunas=predito) [0,0]={cm[0,0]} [0,1]={cm[0,1]} [1,0]={cm[1,0]} [1,1]={cm[1,1]}")
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{nome}\nAcc: {acc:.4f}")
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

**Avançado (com hiperparâmetros)**

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 1. Preparação dos Dados (conforme sua análise)
features = ['Coliformes Termotolerantes', 'Fósforo Total', 'Cor']
X = df_conama_analise_conclusao_final[features].astype(int)
y = df_conama_analise_conclusao_final['dentro'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definição dos Modelos (Dicionário para facilitar o loop)
modelos = {
    "RF_Avancado": RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42),
    "GB_Avancado": GradientBoostingClassifier(n_estimators=300, subsample=0.8, max_features='sqrt', random_state=42),
    "XGB_Avancado": XGBClassifier(n_estimators=500, learning_rate=0.01, max_depth=4, random_state=42)
}

# 3. Treinamento e Exibição de Resultados
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

resultados = {}

for i, (nome, modelo) in enumerate(modelos.items()):
    # Treino
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    resultados[nome] = acc
    
    print(f"\n{'='*30}\nMODELO: {nome}\n{'='*30}")
    print(classification_report(y_test, y_pred, target_names=['Imprópria (0)', 'Própria (1)']))
    
    # Matriz de Confusão Visual
    cm = confusion_matrix(y_test, y_pred)
    print(f"{nome:22s} | Acc: {acc:.4f} | Matriz (linhas=real, colunas=predito) [0,0]={cm[0,0]} [0,1]={cm[0,1]} [1,0]={cm[1,0]} [1,1]={cm[1,1]}")
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{nome}\nAcc: {acc:.4f}")
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

### Fase 2 — entra o clima

Acrescento Radiação, Temperatura do Ar, pH e a contagem de caracteres da conclusão do laudo.


**Simples (baseline)**

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 1. Preparação dos Dados (conforme sua análise)
features_filtro_biologico_clima = ['Coliformes Termotolerantes', 'Fósforo Total', 'Cor', 'ph_limpo', 'RADIACAO', 'conclusao_caracteres', 'TEMP_AR']
X = df_conama_analise_conclusao_final[features_filtro_biologico_clima].astype(int)
y = df_conama_analise_conclusao_final['dentro'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definição dos Modelos (Dicionário para facilitar o loop)
modelos = {
    "RF_Simples": RandomForestClassifier(random_state=42),
    "GB_Simples": GradientBoostingClassifier(random_state=42),
    "XGB_Simples": XGBClassifier(random_state=42),
}

# 3. Treinamento e Exibição de Resultados
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

resultados = {}

for i, (nome, modelo) in enumerate(modelos.items()):
    # Treino
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    resultados[nome] = acc
    
    print(f"\n{'='*30}\nMODELO: {nome}\n{'='*30}")
    print(classification_report(y_test, y_pred, target_names=['Imprópria (0)', 'Própria (1)']))
    
    # Matriz de Confusão Visual
    cm = confusion_matrix(y_test, y_pred)
    print(f"{nome:22s} | Acc: {acc:.4f} | Matriz (linhas=real, colunas=predito) [0,0]={cm[0,0]} [0,1]={cm[0,1]} [1,0]={cm[1,0]} [1,1]={cm[1,1]}")
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{nome}\nAcc: {acc:.4f}")
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

**Avançado (com hiperparâmetros)**

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 1. Preparação dos Dados (conforme sua análise)
features_filtro_biologico_clima = ['Coliformes Termotolerantes', 'Fósforo Total', 'Cor', 'ph_limpo', 'RADIACAO', 'conclusao_caracteres', 'TEMP_AR']
X = df_conama_analise_conclusao_final[features_filtro_biologico_clima].astype(int)
y = df_conama_analise_conclusao_final['dentro'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definição dos Modelos (Dicionário para facilitar o loop)
modelos = {
    "RF_Avancado": RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42),
    "GB_Avancado": GradientBoostingClassifier(n_estimators=300, subsample=0.8, max_features='sqrt', random_state=42),
    "XGB_Avancado": XGBClassifier(n_estimators=500, learning_rate=0.01, max_depth=4, random_state=42)
}

# 3. Treinamento e Exibição de Resultados
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

resultados = {}

for i, (nome, modelo) in enumerate(modelos.items()):
    # Treino
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    resultados[nome] = acc
    
    print(f"\n{'='*30}\nMODELO: {nome}\n{'='*30}")
    print(classification_report(y_test, y_pred, target_names=['Imprópria (0)', 'Própria (1)']))
    
    # Matriz de Confusão Visual
    cm = confusion_matrix(y_test, y_pred)
    print(f"{nome:22s} | Acc: {acc:.4f} | Matriz (linhas=real, colunas=predito) [0,0]={cm[0,0]} [0,1]={cm[0,1]} [1,0]={cm[1,0]} [1,1]={cm[1,1]}")
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{nome}\nAcc: {acc:.4f}")
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

### Fase 3 — Filtro Biológico Secundário

Acrescento Amônia Total e DBO.


**Simples (baseline)**

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 1. Preparação dos Dados (conforme sua análise)
features = ['Coliformes Termotolerantes', 'Fósforo Total', 'Cor', 'ph_limpo', 
                                   'RADIACAO', 'conclusao_caracteres', 'TEMP_AR', 'Amônia Total', 'DBO']
X = df_conama_analise_conclusao_final[features].astype(int)
y = df_conama_analise_conclusao_final['dentro'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definição dos Modelos (Dicionário para facilitar o loop)
modelos = {
    "RF_Simples": RandomForestClassifier(random_state=42),
    "GB_Simples": GradientBoostingClassifier(random_state=42),
    "XGB_Simples": XGBClassifier(random_state=42),
}

# 3. Treinamento e Exibição de Resultados
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

resultados = {}

for i, (nome, modelo) in enumerate(modelos.items()):
    # Treino
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    resultados[nome] = acc
    
    print(f"\n{'='*30}\nMODELO: {nome}\n{'='*30}")
    print(classification_report(y_test, y_pred, target_names=['Imprópria (0)', 'Própria (1)']))
    
    # Matriz de Confusão Visual
    cm = confusion_matrix(y_test, y_pred)
    print(f"{nome:22s} | Acc: {acc:.4f} | Matriz (linhas=real, colunas=predito) [0,0]={cm[0,0]} [0,1]={cm[0,1]} [1,0]={cm[1,0]} [1,1]={cm[1,1]}")
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{nome}\nAcc: {acc:.4f}")
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

**Avançado (com hiperparâmetros)**

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# 1. Preparação dos Dados (conforme sua análise)
features = ['Coliformes Termotolerantes', 'Fósforo Total', 'Cor', 'ph_limpo', 
                                   'RADIACAO', 'conclusao_caracteres', 'TEMP_AR', 'Amônia Total', 'DBO']
X = df_conama_analise_conclusao_final[features].astype(int)
y = df_conama_analise_conclusao_final['dentro'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definição dos Modelos (Dicionário para facilitar o loop)
modelos = {
    "RF_Avancado": RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42),
    "GB_Avancado": GradientBoostingClassifier(n_estimators=300, subsample=0.8, max_features='sqrt', random_state=42),
    "XGB_Avancado": XGBClassifier(n_estimators=500, learning_rate=0.01, max_depth=4, random_state=42)    
}

# 3. Treinamento e Exibição de Resultados
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

resultados = {}

for i, (nome, modelo) in enumerate(modelos.items()):
    # Treino
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    resultados[nome] = acc
    
    print(f"\n{'='*30}\nMODELO: {nome}\n{'='*30}")
    print(classification_report(y_test, y_pred, target_names=['Imprópria (0)', 'Própria (1)']))
    
    # Matriz de Confusão Visual
    cm = confusion_matrix(y_test, y_pred)
    print(f"{nome:22s} | Acc: {acc:.4f} | Matriz (linhas=real, colunas=predito) [0,0]={cm[0,0]} [0,1]={cm[0,1]} [1,0]={cm[1,0]} [1,1]={cm[1,1]}")
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{nome}\nAcc: {acc:.4f}")
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

**Super Otimizada**

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Preparação dos Dados (Fase 3 Completa)
features = ['Coliformes Termotolerantes', 'Fósforo Total', 'Cor', 'ph_limpo', 
            'RADIACAO', 'conclusao_caracteres', 'TEMP_AR', 'Amônia Total', 'DBO']

X = df_conama_analise_conclusao_final[features].astype(int)
y = df_conama_analise_conclusao_final['dentro'].astype(int)

# Mantendo a mesma divisão para comparação justa
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definição dos Modelos Super Otimizados
# O segredo aqui é o equilíbrio entre profundidade e aleatoriedade (subsample)
modelos = {
    "RF_Super_Otimizado": RandomForestClassifier(
        n_estimators=500,           # Mais árvores para reduzir a variância
        max_depth=None,             # Deixamos crescer, mas controlamos pelas folhas
        min_samples_leaf=2,         # Evita folhas com apenas 1 dado (ruído)
        max_features='sqrt', 
        bootstrap=True, 
        n_jobs=-1,                  # Usa todos os núcleos do processador
        random_state=42
    ),
    "GB_Super_Otimizado": GradientBoostingClassifier(
        n_estimators=600, 
        learning_rate=0.08,         # Taxa moderada para não pular o mínimo global
        max_depth=6,                # Um pouco mais que o Baseline, menos que o infinito
        subsample=0.85,             # Treina cada árvore com 85% dos dados (evita decoras)
        max_features='sqrt',
        random_state=42
    ),
    "XGB_Super_Otimizado": XGBClassifier(
        n_estimators=1000, 
        learning_rate=0.05, 
        max_depth=8,                # Permite capturar interações complexas entre Clima e Química
        gamma=0.2,                  # Poda árvores que não trazem ganho real
        subsample=0.8,              # Aleatoriedade nas linhas
        colsample_bytree=0.8,       # Aleatoriedade nas colunas (features)
        reg_alpha=0.1,              # Regularização L1 para evitar pesos exagerados
        random_state=42
    )
}

# 3. Treinamento e Exibição de Resultados
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes = axes.flatten()

for i, (nome, modelo) in enumerate(modelos.items()):
    # Treino
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n{'='*40}\nMODELO: {nome}\n{'='*40}")
    print(classification_report(y_test, y_pred, target_names=['Imprópria (0)', 'Própria (1)']))
    
    # Matriz de Confusão Visual
    cm = confusion_matrix(y_test, y_pred)
    print(f"{nome:22s} | Acc: {acc:.4f} | Matriz (linhas=real, colunas=predito) [0,0]={cm[0,0]} [0,1]={cm[0,1]} [1,0]={cm[1,0]} [1,1]={cm[1,1]}")
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[i], cbar=False) # Mudei para verde para destacar a otimização
    axes[i].set_title(f"{nome}\nAcc: {acc:.4f}")
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

# 9. Evolução dos modelos XGBoost, Randon Forest e o Gradiente Descendente com apenas as **Features Filtro Biologico** **Clima** **Features Biologicas Secundarias**

Explicação deste gráfico em três momentos:

1. O Platô Inicial (Filtro Biológico): "No início, com apenas as 3 variáveis básicas (Coliformes, Fósforo e Cor), todos os modelos estagnaram em 80%. Eles tinham o 'vício' de serem simplistas demais, gerando muitos falsos positivos."

2. O Salto Climático (Clima): "O grande salto de performance (de 80% para 91%+) aconteceu quando ensinamos o modelo a 'olhar para o céu'. A inclusão de Radiação e Temperatura do Ar permitiu que o algoritmo entendesse o contexto ambiental da poluição."

3. O Refinamento Final (Secundárias): "Por fim, a Amônia e a DBO trouxeram a estabilidade final. Embora o aumento na acurácia pareça pequeno no gráfico, ele representa a eliminação de casos críticos de contaminação química, tornando o modelo pronto para o mundo real."

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Dados
dados_evolucao = {
    'Etapa': ['Filtro Biológico', 'Clima', 'Secundárias (Amônia/DBO)'],
    'Random Forest': [0.8031, 0.9100, 0.9106],
    'Gradient Boosting': [0.8031, 0.9232, 0.9235],
    'XGBoost': [0.8031, 0.9122, 0.9130]
}

df_evolucao = pd.DataFrame(dados_evolucao)
df_melted = df_evolucao.melt(id_vars='Etapa', var_name='Modelo', value_name='Acurácia')

# Estética
plt.figure(figsize=(14, 7))
sns.set_style("whitegrid", {'axes.facecolor': '#f9f9f9'})

# Paleta de cores vibrante
paleta = ["#34495e", "#e67e22", "#27ae60"]

# Plotagem principal
lineplot = sns.lineplot(data=df_melted, x='Etapa', y='Acurácia', hue='Modelo', 
                        style='Modelo', markers=True, dashes=False, 
                        markersize=12, linewidth=4, palette=paleta)

# Melhorando o Eixo Y para mostrar o crescimento
plt.ylim(0.78, 0.95)
plt.yticks([0.80, 0.85, 0.90, 0.925, 0.95], ['80%', '85%', '90%', '92.5%', '95%'])

# Anotações inteligentes para evitar sobreposição no final
offset = {
    'Gradient Boosting': 0.005,
    'XGBoost': 0.000,
    'Random Forest': -0.005
}

for modelo in df_melted['Modelo'].unique():
    val = df_melted[df_melted['Modelo'] == modelo]['Acurácia'].iloc[-1]
    plt.text(2.05, val + offset[modelo], f'{modelo}: {val:.2%}', 
             fontsize=11, fontweight='bold', color=lineplot.get_lines()[df_melted['Modelo'].unique().tolist().index(modelo)].get_color())

# Títulos e Rótulos
plt.title('Evolução da Inteligência do Modelo: O Impacto das Camadas de Dados', fontsize=18, fontweight='bold', loc='left', pad=25)
plt.ylabel('Acurácia do Modelo', fontsize=13, fontweight='semibold')
plt.xlabel('Etapas de Incremento de Dados', fontsize=13, fontweight='semibold')

# Adicionando uma seta indicando o maior salto
plt.annotate('Maior ganho: +11% de Acurácia\n(Inclusão de Variáveis Climáticas)', 
             xy=(1, 0.915), xytext=(0.4, 0.93),
             arrowprops=dict(facecolor='black', shrink=0.05, width=2),
             fontsize=10, bbox=dict(boxstyle="round", fc="0.9"))

plt.legend(title='Arquitetura do Modelo', title_fontsize='12', loc='upper left')
plt.tight_layout()
plt.show()

### 9.1 Evolução dos modelos XGBoost, Randon Forest e o Gradiente Descendente com Implementação Simples 
* **Features Filtro Biologico** 
* **Clima** 
* **Features Biologicas Secundarias**

In [ ]:
# Dados
dados_evolucao = {
    'Etapa': ['Filtro Biológico', 'Clima', 'Secundárias (Amônia/DBO)'],
    'Random Forest': [0.8031, 0.9516, 0.9530],
    'Gradient Boosting': [0.8031, 0.9166, 0.9162],
    'XGBoost': [0.8031, 0.9465, 0.9484]
}

df_evolucao = pd.DataFrame(dados_evolucao)
df_melted = df_evolucao.melt(id_vars='Etapa', var_name='Modelo', value_name='Acurácia')

# Estética
plt.figure(figsize=(14, 7))
sns.set_style("whitegrid", {'axes.facecolor': '#f9f9f9'})

# Paleta de cores vibrante
paleta = ["#34495e", "#e67e22", "#27ae60"]

# Plotagem principal
lineplot = sns.lineplot(data=df_melted, x='Etapa', y='Acurácia', hue='Modelo', 
                        style='Modelo', markers=True, dashes=False, 
                        markersize=12, linewidth=4, palette=paleta)

# Melhorando o Eixo Y para mostrar o crescimento
plt.ylim(0.78, 0.97)
plt.yticks([0.80, 0.85, 0.90, 0.925, 0.95, 0.97], ['80%', '85%', '90%', '92.5%', '95%', '97%'])

# Anotações inteligentes para evitar sobreposição no final
offset = {
    'Gradient Boosting': -0.001,
    'XGBoost': -0.002,
    'Random Forest': 0.000
}

for modelo in df_melted['Modelo'].unique():
    val = df_melted[df_melted['Modelo'] == modelo]['Acurácia'].iloc[-1]
    plt.text(2.05, val + offset[modelo], f'{modelo}: {val:.2%}', 
             fontsize=11, fontweight='bold', color=lineplot.get_lines()[df_melted['Modelo'].unique().tolist().index(modelo)].get_color())

# Títulos e Rótulos
plt.title('Evolução da Inteligência do Modelo: O Impacto das Camadas de Dados', fontsize=18, fontweight='bold', loc='left', pad=25)
plt.ylabel('Acurácia do Modelo', fontsize=13, fontweight='semibold')
plt.xlabel('Etapas de Incremento de Dados', fontsize=13, fontweight='semibold')

# Adicionando uma seta indicando o maior salto
plt.annotate('Maior ganho: +13% de Acurácia\n(Inclusão de Variáveis Climáticas)', 
             xy=(1, 0.951), xytext=(0.4, 0.95),
             arrowprops=dict(facecolor='black', shrink=0.05, width=2),
             fontsize=10, bbox=dict(boxstyle="round", fc="0.9"))

plt.legend(title='Arquitetura do Modelo', title_fontsize='12', loc='upper left')
plt.tight_layout()
plt.show()

## 10. Antes e depois da correção

Os números de "Antes" são os que eu já tinha mostrado no Draft-T9, com o bug. Deixei eles aqui
só pra comparação — os de "Depois" são os que saem rodando o notebook corrigido, acima.


In [ ]:
comparacao = pd.DataFrame([
    # Fase, Config, Modelo, Acc_Antes, Acc_Depois
    ["1 - Biologico",      "Baseline",       "RF",  0.8031, 0.7895],
    ["1 - Biologico",      "Baseline",       "GB",  0.8031, 0.7895],
    ["1 - Biologico",      "Baseline",       "XGB", 0.8031, 0.7895],
    ["2 - Bio+Clima",      "Baseline",       "RF",  0.9516, 0.9608],
    ["2 - Bio+Clima",      "Baseline",       "GB",  0.9166, 0.9418],
    ["2 - Bio+Clima",      "Baseline",       "XGB", 0.9486, 0.9648],
    ["2 - Bio+Clima",      "Avancado",       "RF",  0.9099, 0.9022],
    ["2 - Bio+Clima",      "Avancado",       "GB",  0.9232, 0.9518],
    ["2 - Bio+Clima",      "Avancado",       "XGB", 0.9121, 0.9351],
    ["3 - Completa",       "Baseline",       "RF",  0.9530, 0.9599],
    ["3 - Completa",       "Baseline",       "GB",  0.9162, 0.9418],
    ["3 - Completa",       "Baseline",       "XGB", 0.9483, 0.9630],
    ["3 - Completa",       "Avancado",       "RF",  0.9106, 0.9044],
    ["3 - Completa",       "Avancado",       "GB",  0.9235, 0.9504],
    ["3 - Completa",       "Avancado",       "XGB", 0.9129, 0.9310],
    ["3 - Completa",       "Super Otimizado","RF",  0.9530, 0.9603],
    ["3 - Completa",       "Super Otimizado","GB",  0.9537, 0.9648],
    ["3 - Completa",       "Super Otimizado","XGB", 0.9594, 0.9662],
], columns=["Fase", "Configuracao", "Modelo", "Acc_Antes_vazado", "Acc_Depois_corrigido"])

comparacao["Diferenca_pp"] = ((comparacao["Acc_Depois_corrigido"] - comparacao["Acc_Antes_vazado"]) * 100).round(2)
for col in ["Acc_Antes_vazado", "Acc_Depois_corrigido"]:
    comparacao[col] = (comparacao[col] * 100).round(2)
comparacao


## 11. O que ficou depois da correção

A conclusão do TCC não muda: o salto de acurácia continua acontecendo bem na passagem da Fase 1
pra Fase 2 (quando entra o clima), e o XGBoost Super Otimizado da Fase 3 continua sendo o
melhor modelo — só que agora com **96,62%** (era 95,94% com o bug), em cima de um teste de
2.218 laudos de verdade, não 19.372.

E a Fase 1 ficou até mais clara depois da correção: as 6 combinações de modelo e configuração
dão exatamente a mesma matriz de confusão. Isso não é efeito da duplicação — é porque com só 3
variáveis binárias não sobra muito espaço pro modelo decidir diferente.
